# KL-Anchor Validation vs Pure Population-$\beta$ Loss (`jaredpalmer/kev-9b`)

Tests whether adding forward $\mathrm{KL}(p_0 \,\|\, p_\theta)$ toward the frozen base model stabilizes pure table-only training on `wu2025` (`8` fitted OLS slopes, `0` respondent choices $y$):

$$\mathcal{L}(\theta) = \lambda\,\widehat{\mathcal{L}}_{\text{pop-}\beta}^{\text{split}}(p_\theta, \beta^*_{\text{Wu}}) + \alpha\,\mathrm{KL}(p_0 \,\|\, p_\theta), \qquad \lambda = 1.0$$

Because Wu's 8 slopes constrain only an 8-dimensional projection of the model's choice probabilities across `18,591` tasks, the remaining directions are unconstrained when $\alpha = 0$. On Kev-0.8B, in `v13` (`α = 0`, `lr = 1e-5`), `kriner2026/beta_mse` improved (`0.01775 -> 0.00396`), while `wu2025` `monitor_kl` drifted (`0.83 -> 2.27`) and held-out `grummon2025/beta_mse` degraded (`0.00546 -> 0.00844` at `best.pt`, spiking to `0.0985` at step 500).

| Run Folder (`runs/kl_anchor_9b/`) | $\lambda$ | $\alpha$ (`kl_anchor_alpha`) | Peak LR | Epochs | Steps | Role |
|---|---:|---:|---:|---:|---:|---|
| `wu_table_alpha=0` | 1.0 | `0.0` | `5e-6` | 3 | 1,395 | Pure split population-$\beta$ baseline (no KL anchor) |
| `wu_table_alpha=0.02` | 1.0 | `0.02` | `5e-6` | 3 | 1,395 | Light KL anchor |
| `wu_table_alpha=0.1` | 1.0 | `0.10` | `5e-6` | 3 | 1,395 | Moderate KL anchor |
| `wu_table_alpha=0.5` | 1.0 | `0.50` | `5e-6` | 3 | 1,395 | Strong KL anchor |

All outputs are saved under `ms_paper_beta_loss/runs/kl_anchor_9b/` so the Kev-0.8B sweep in `runs/kl_anchor_800m/` (`00-validate-kl-anchor-800m.ipynb`) remains untouched. Checkpoint selection uses `BEST_METRIC = "kriner2026/beta_mse"`; `grummon2025` is held out for evaluation only.

In [ ]:
import os
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

# Walk up from the notebook folder to locate the repository's `code/` root
CODE_DIR = next(
    (c for p in (Path.cwd(), *Path.cwd().parents) for c in (p, p / "code") if all((c / d).is_dir() for d in ("kev", "data", "training"))),
    None,
)
if CODE_DIR is None:
    raise RuntimeError(f"No code/ root containing kev/, data/, and training/ at or above {Path.cwd()}")

BETA_TRAINING_DIR = CODE_DIR / "train-notebooks" / "beta-training"
PAPER_DIR = BETA_TRAINING_DIR / "ms_paper_beta_loss"
if not (PAPER_DIR / "paper_beta").is_dir():
    raise RuntimeError(f"`paper_beta` package not found in {PAPER_DIR}")

# Run from `code/` so data loaders and shared disk caches resolve identically to all beta-training notebooks
os.chdir(CODE_DIR)
for _p in (str(CODE_DIR), str(CODE_DIR / "kev"), str(BETA_TRAINING_DIR), str(PAPER_DIR)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from models.kev_pointer import KevPointerModel
from training import free_gpu

import beta_ols
from beta_ols import core as beta_core
from beta_train import runs as run_store
from beta_train.diagnostics import find_max_micro_chunk
from beta_train.items import build_items_cached, items_summary
from beta_train.model_state import set_checkpointing, snapshot_trainable
from beta_train.validation import PopulationValidator, validate_all_cached, validation_metrics_frame
from paper_beta import (
    PaperTrainConfig,
    extract_step0_from_run_dir,
    load_paper_run,
    run_paper_training,
)
from paper_beta.trainer import paper_steps_per_epoch

# Shared disk cache for tokenized items and untrained step-0 validation; isolated run directory for alpha validation
CACHE_DIR = CODE_DIR / ".cache" / "beta_training"
RUNS_DIR = PAPER_DIR / "runs" / "kl_anchor_9b"

def alpha_slug(alpha: float) -> str:
    return f"wu_table_alpha={alpha:g}"


print(f"CODE_DIR : {CODE_DIR}")
print(f"PAPER_DIR: {PAPER_DIR}")
print(f"RUNS_DIR : {RUNS_DIR}")

In [ ]:
# Require CUDA and release any previously loaded `kev_model` before allocating a fresh 9B backbone
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required to train jaredpalmer/kev-9b in bf16")
if "kev_model" in globals():
    globals().pop("kev_model")
device = "cuda"
free_gpu()
torch.manual_seed(42)

KEV_BASE = "jaredpalmer/kev-9b@db029f08b290afd9fee4aa4bbcd9ae48602d1eb0"
LORA_RANK = 64
WEIGHTS_DTYPE = "bf16"
HEAD_TEMPERATURE = 1.0
GRADIENT_CHECKPOINTING = False

t_model0 = time.perf_counter()
kev_model = KevPointerModel.build(
    KEV_BASE,
    device=device,
    lora_rank=LORA_RANK,
    weights_dtype=WEIGHTS_DTYPE,
)
kev_model.net.head.temperature = HEAD_TEMPERATURE
kev_model.train()
set_checkpointing(kev_model, GRADIENT_CHECKPOINTING)
torch.cuda.synchronize()
model_load_sec = time.perf_counter() - t_model0

trainable_params_count = sum(p.numel() for p in kev_model.trainable_parameters())
total_params_count = sum(p.numel() for p in kev_model.parameters())
display(
    pd.Series(
        {
            "base_model": KEV_BASE,
            "mode": kev_model.mode,
            "training": kev_model.training,
            "weights_dtype": kev_model.meta.weights_dtype,
            "lora_rank": kev_model.meta.lora,
            "trainable_params": f"{trainable_params_count:,}",
            "total_params": f"{total_params_count:,}",
            "gradient_checkpointing": GRADIENT_CHECKPOINTING,
            "model_load_sec": round(model_load_sec, 3),
        },
        name="kev_9b_train_mode",
    ).to_frame()
)

INIT_TRAINABLE_STATE = snapshot_trainable(kev_model)
MODEL_KEY = {
    "base": KEV_BASE,
    "lora_rank": LORA_RANK,
    "weights_dtype": WEIGHTS_DTYPE,
    "head_temperature": HEAD_TEMPERATURE,
}
print(f"Snapshotted {len(INIT_TRAINABLE_STATE)} trainable tensors")

# 1. `wu2025` Fitted OLS Targets & Held-Out Datasets (`kriner2026`, `grummon2025`)

In [ ]:
ITEM_DATASETS = ["wu2025"]
VALIDATION_DATASETS = ["wu2025", "kriner2026", 'grummon2025']
LOADED_DATASETS = [*ITEM_DATASETS, *VALIDATION_DATASETS]

choice_dfs: dict[str, pd.DataFrame] = {}
regression_rows_by_dataset: dict[str, pd.DataFrame] = {}
regression_specs_by_dataset: dict[str, dict] = {}
human_ols_by_dataset: dict[str, pd.DataFrame] = {}
module_by_dataset = {}
piece_names_by_study: dict[str, tuple[str, ...]] = {}

for dataset_name in LOADED_DATASETS:
    dataset_module = beta_ols.DATASET_MODULES[dataset_name]
    full_choice_df = dataset_module.load_choice_df()
    pieces = {}
    if dataset_name in VALIDATION_DATASETS:
        pieces[dataset_name] = full_choice_df
    if dataset_name in ITEM_DATASETS:
        task_pieces = beta_core.split_by_task(full_choice_df, dataset_name)
        pieces.update(task_pieces)
        piece_names_by_study[dataset_name] = tuple(task_pieces)

    for piece_name, choice_df in pieces.items():
        rows = dataset_module.regression_rows(choice_df)
        specs = dataset_module.regression_specs(rows)
        module_by_dataset[piece_name] = dataset_module
        choice_dfs[piece_name] = choice_df
        regression_rows_by_dataset[piece_name] = rows
        regression_specs_by_dataset[piece_name] = specs
        human_ols_by_dataset[piece_name] = beta_core.fit_human_ols(rows, specs)
        print(
            f"{piece_name}: {choice_df['CASEID'].nunique():,} respondents | "
            f"{len(choice_df):,} tasks | {len(specs)} regression(s)"
        )
        display(beta_core.wide_ols_table(human_ols_by_dataset[piece_name]).round(4))

WU_DATASETS = piece_names_by_study["wu2025"]

# Table targets for `wu2025`: the 8 standardized OLS slopes fitted on Wu's choices, standing in for a published table
# (Wu's respondent choices y never enter the training loss or gradient)
target_std_betas_by_dataset: dict[str, dict[str, torch.Tensor]] = {}
for dataset_name in WU_DATASETS:
    dataset_ols = human_ols_by_dataset[dataset_name]
    dataset_targets = {}
    for group, spec in regression_specs_by_dataset[dataset_name].items():
        group_ols = dataset_ols[dataset_ols["model"] == group].set_index("term")
        target_values = group_ols.loc[spec.target_labels, "std_beta"].to_numpy(dtype="float64")
        dataset_targets[group] = torch.tensor(target_values, dtype=torch.float64)
        display(
            pd.Series(
                target_values,
                index=spec.target_labels,
                name=f"{dataset_name} | {group} fitted OLS target std_beta",
            ).round(4).to_frame()
        )
    target_std_betas_by_dataset[dataset_name] = dataset_targets

print(f"WU_DATASETS = {WU_DATASETS}")

# 2. Validators, Cached Step-0 Evaluation, & Cached `wu2025` Items

In [ ]:
VALIDATION_MAX_QUESTIONS = 1024
VALIDATION_MAX_CHARS = 1_000_000

validators = {
    dataset_name: PopulationValidator(
        beta_ols.DATASET_MODULES[dataset_name],
        choice_dfs[dataset_name],
        regression_specs_by_dataset[dataset_name],
        max_questions=VALIDATION_MAX_QUESTIONS,
        max_chars=VALIDATION_MAX_CHARS,
    )
    for dataset_name in VALIDATION_DATASETS
}
display(pd.DataFrame([v.describe() for v in validators.values()]).set_index("validator"))

In [ ]:
# Untrained Kev-9B evaluation (shared disk cache, keyed by MODEL_KEY)
step0_validation, step0_cache_report = validate_all_cached(
    kev_model,
    validators,
    INIT_TRAINABLE_STATE,
    model_key=MODEL_KEY,
    cache_dir=CACHE_DIR,
)
display(step0_cache_report)
display(validation_metrics_frame(step0_validation).round(5))

In [ ]:
item_modules = {dataset_name: module_by_dataset[dataset_name] for dataset_name in WU_DATASETS}
unified_items, items_cache_report = build_items_cached(
    kev_model, item_modules, choice_dfs, cache_dir=CACHE_DIR
)
display(items_cache_report)
display(items_summary(unified_items, regression_specs_by_dataset))

# 3. Shared Training Settings & Per-$\alpha$ Runner

Matches the Kev-0.8B sweep in `00-validate-kl-anchor-800m.ipynb` except `PEAK_LR = 5e-6` (the Kev-4B rate in `02-train-paper-4b.ipynb`) (`S = 40`, `3` epochs = `1,395` steps, `beta_lambda = 1.0`), varying only `kl_anchor_alpha` across `(0.0, 0.02, 0.1, 0.5)`. Validation runs every `50` steps on `kriner2026` (selection) and once per epoch (`465` steps) plus `best.pt` on held-out `grummon2025`.

In [ ]:
MAX_FITTING_MICRO_CHUNK = 16
RUN_MICRO_CHUNK_SEARCH = True

if RUN_MICRO_CHUNK_SEARCH:
    search_items = [item for ds in WU_DATASETS for item in unified_items[ds]]
    micro_chunk_search_df, MAX_FITTING_MICRO_CHUNK = find_max_micro_chunk(
        kev_model,
        search_items,
        INIT_TRAINABLE_STATE,
        max_chunk=41,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        candidates_per_size=3,
    )

    if MAX_FITTING_MICRO_CHUNK>1: MAX_FITTING_MICRO_CHUNK-=1 # for safety decrease microchunk by 1
    display(micro_chunk_search_df)

NOTEBOOK = "00-validate-kl-anchor-9b"
BEST_METRIC = "kriner2026/beta_mse"
BEST_MODE = "min"

from_scratch = False

SAMPLE_SIZE = 40
PEAK_LR = 5e-6
NUM_EPOCHS = 3
BETA_LAMBDA = 1.0
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
SEED = 42
MICRO_CHUNK = MAX_FITTING_MICRO_CHUNK
P0_CHUNK = 32

EVAL_EVERY_BY_VALIDATOR = {
    "wu2025": 50,    # 28 evaluations across 1,395 steps (~8 min total)
    "kriner2026": 50,    # 28 evaluations across 1,395 steps (~8 min total)
    "grummon2025": 465,  # end of each Wu epoch (steps 465, 930, 1,395) + step 0 + best.pt
}

RUN_METADATA = {
    "notebook": NOTEBOOK,
    "model": MODEL_KEY,
    "item_datasets": ITEM_DATASETS,
    "validation_datasets": VALIDATION_DATASETS,
    "validator_kind": "fused",
}
runs = {}


def make_kl_anchor_config(kl_anchor_alpha: float) -> PaperTrainConfig:
    return PaperTrainConfig(
        ce_datasets=(),
        beta_datasets=WU_DATASETS,
        sample_size_by_dataset={ds: SAMPLE_SIZE for ds in WU_DATASETS},
        beta_lambda=BETA_LAMBDA,
        kl_anchor_alpha=kl_anchor_alpha,
        lr=PEAK_LR,
        num_epochs=NUM_EPOCHS,
        eval_every_by_validator=EVAL_EVERY_BY_VALIDATOR,
        lr_schedule="onecycle",
        weight_decay=WEIGHT_DECAY,
        micro_chunk=MICRO_CHUNK,
        p0_chunk=P0_CHUNK,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        grad_clip=GRAD_CLIP,
        seed=SEED,
    )


def train_kl_anchor_run(kl_anchor_alpha: float):
    slug = alpha_slug(kl_anchor_alpha)
    run_config = make_kl_anchor_config(kl_anchor_alpha)
    print(f"=== {slug} (beta_lambda={BETA_LAMBDA}, kl_anchor_alpha={kl_anchor_alpha}, lr={PEAK_LR}, epochs={NUM_EPOCHS}) ===")
    display(pd.Series(run_config.as_dict(), name=slug).to_frame())
    runs[slug] = run_paper_training(
        model=kev_model,
        items_by_dataset=unified_items,
        specs_by_dataset=regression_specs_by_dataset,
        target_std_betas_by_dataset=target_std_betas_by_dataset,
        validators=validators,
        step0_validation=step0_validation,
        init_snapshot=INIT_TRAINABLE_STATE,
        config=run_config,
        device=device,
        run_dir=RUNS_DIR / slug,
        best_metric=BEST_METRIC,
        best_mode=BEST_MODE,
        run_metadata={**RUN_METADATA, "kl_anchor_alpha": kl_anchor_alpha, "sweep_slug": slug},
        from_scratch=from_scratch,
    )
    return runs[slug]


In [ ]:
ALPHA_SWEEP: tuple[float, ...] = (0.0, 0.02, 0.1, 0.5)

schedule_rows = []
for _alpha in ALPHA_SWEEP:
    _cfg = make_kl_anchor_config(_alpha)
    _ep_steps = paper_steps_per_epoch(unified_items, _cfg)
    schedule_rows.append(
        {
            "slug": alpha_slug(_alpha),
            "beta_lambda": _cfg.beta_lambda,
            "kl_anchor_alpha": _cfg.kl_anchor_alpha,
            "lr": _cfg.lr,
            "epochs": _cfg.num_epochs,
            "steps_per_epoch": _ep_steps,
            "total_steps": _ep_steps * _cfg.num_epochs,
        }
    )
display(pd.DataFrame(schedule_rows).set_index("slug"))

## Run 1 — Pure Population-$\beta$ Baseline (`kl_anchor_alpha = 0.0`)

In [ ]:
train_kl_anchor_run(0.0)

## Run 2 — Light KL Anchor (`kl_anchor_alpha = 0.02`)

In [ ]:
train_kl_anchor_run(0.02)

## Run 3 — Moderate KL Anchor (`kl_anchor_alpha = 0.1`)

In [ ]:
train_kl_anchor_run(0.1)

## Run 4 — Strong KL Anchor (`kl_anchor_alpha = 0.5`)

In [ ]:
train_kl_anchor_run(0.5)

# 4. Comparison Table & Optimization Diagnostics (`runs/kl_anchor_9b/`)

Reads directly from disk (requires only Cell 1 imports, so it can be run on CPU while or after training finishes). Compares both **`best.pt` (selected by `kriner2026/beta_mse`)** and **final-step (`step 1,395`) stability** across $\alpha \in \{0.0, 0.02, 0.1, 0.5\}$.

In [ ]:
def _sign_acc(beta_df: pd.DataFrame) -> float:
    return float(np.mean(np.sign(beta_df["model_pop_beta"].to_numpy(dtype=float)) == np.sign(beta_df["human_pop_beta"].to_numpy(dtype=float))))


def summarize_kl_anchor_sweep(runs_dir: Path, alphas: tuple[float, ...] = ALPHA_SWEEP) -> pd.DataFrame:
    runs_dir = Path(runs_dir)
    validators_pair = ("kriner2026", "grummon2025")
    first_existing = next((runs_dir / alpha_slug(a) for a in alphas if (runs_dir / alpha_slug(a) / "run.json").exists()), None)
    rows = []

    if first_existing is not None:
        base_info = extract_step0_from_run_dir(first_existing, validators_pair)
        m0 = base_info["metrics"]
        rows.append(
            {
                "slug": "step0_untrained_base",
                "kl_alpha": float("nan"),
                "status": "cached_step0",
                "best_step": 0,
                "last_step": 0,
                "kriner_best_mse": m0["kriner2026/beta_mse"],
                "kriner_best_rho": m0["kriner2026/spearman_rho"],
                "kriner_best_kl": m0["kriner2026/kl_loss"],
                "grummon_best_mse": m0["grummon2025/beta_mse"],
                "grummon_best_rho": m0["grummon2025/spearman_rho"],
                "grummon_best_sign": m0["grummon2025/sign_acc"],
                "grummon_best_kl": m0["grummon2025/kl_loss"],
                "kriner_final_mse": m0["kriner2026/beta_mse"],
                "grummon_final_mse": m0["grummon2025/beta_mse"],
                "wu_cross_last15": float("nan"),
                "wu_kl_p0_last15": 0.0,
                "wu_sample_mse_last15": float("nan"),
            }
        )

    for alpha in alphas:
        slug = alpha_slug(alpha)
        run_dir = runs_dir / slug
        if not (run_dir / "run.json").exists():
            rows.append({"slug": slug, "kl_alpha": alpha, "status": "not_run_yet"})
            continue
        run_json = run_store.read_run_json(run_dir)
        res = load_paper_run(run_dir)
        last_val_step = int(res.val_history_df.index[-1])
        last_val_row = res.val_history_df.loc[last_val_step]
        wu_ds = res.per_dataset_df.loc["wu2025"] if "wu2025" in res.per_dataset_df.index else None
        grummon_sign = _sign_acc(res.best_validation["grummon2025"][1]) if "grummon2025" in res.best_validation else float("nan")
        rows.append(
            {
                "slug": slug,
                "kl_alpha": alpha,
                "status": run_json["status"],
                "best_step": int(res.best["step"]),
                "last_step": int(res.step_df["step"].max()) if len(res.step_df) > 0 else 0,
                "kriner_best_mse": float(res.best_row["kriner2026/beta_mse"]),
                "kriner_best_rho": float(res.best_row["kriner2026/spearman_rho"]),
                "kriner_best_kl": float(res.best_row["kriner2026/kl_loss"]),
                "grummon_best_mse": float(res.best_row["grummon2025/beta_mse"]),
                "grummon_best_rho": float(res.best_row["grummon2025/spearman_rho"]),
                "grummon_best_sign": grummon_sign,
                "grummon_best_kl": float(res.best_row["grummon2025/kl_loss"]),
                "kriner_final_mse": float(last_val_row["kriner2026/beta_mse"]),
                "grummon_final_mse": float(last_val_row["grummon2025/beta_mse"]),
                "wu_cross_last15": float(wu_ds["cross_loss_last15pct"]) if wu_ds is not None else float("nan"),
                "wu_kl_p0_last15": float(wu_ds["kl_anchor_last15pct"]) if wu_ds is not None else float("nan"),
                "wu_sample_mse_last15": float(wu_ds["sample_beta_mse_last15pct"]) if wu_ds is not None else float("nan"),
            }
        )
    return pd.DataFrame(rows).set_index("slug")


sweep_summary_df = summarize_kl_anchor_sweep(RUNS_DIR)
display(sweep_summary_df.round(5))

In [ ]:
available_runs = [
    (alpha, load_paper_run(RUNS_DIR / alpha_slug(alpha)))
    for alpha in ALPHA_SWEEP
    if (RUNS_DIR / alpha_slug(alpha) / "run.json").exists()
]

if not available_runs:
    print(f"No runs saved in {RUNS_DIR} yet.")
else:
    fig, axes = plt.subplots(2, 3, figsize=(16, 8.5))
    rolling_window = 30

    for alpha, res in available_runs:
        label = f"alpha={alpha:g} (best@{int(res.best['step'])})"
        st = res.step_df
        vh = res.val_history_df

        if len(st) > 0:
            axes[0, 0].plot(st["step"], st["cross_loss"].rolling(rolling_window, min_periods=1).mean(), linewidth=1.6, label=label)
            axes[0, 1].plot(st["step"], st["kl_anchor_loss"].rolling(rolling_window, min_periods=1).mean(), linewidth=1.6, label=label)
            axes[0, 2].plot(st["step"], st["sample_beta_mse"].rolling(rolling_window, min_periods=1).mean(), linewidth=1.6, label=label)

        for ax, col in (
            (axes[1, 0], "kriner2026/beta_mse"),
            (axes[1, 1], "grummon2025/beta_mse"),
            (axes[1, 2], "grummon2025/kl_loss"),
        ):
            series = vh[col].dropna()
            ax.plot(series.index, series.values, marker="o", markersize=3.5, linewidth=1.6, label=label)

    step0_vh = available_runs[0][1].val_history_df.loc[0]
    axes[1, 0].axhline(float(step0_vh["kriner2026/beta_mse"]), color="gray", linestyle="--", linewidth=1.0, label="Step 0 base")
    axes[1, 1].axhline(float(step0_vh["grummon2025/beta_mse"]), color="gray", linestyle="--", linewidth=1.0, label="Step 0 base")
    axes[1, 2].axhline(float(step0_vh["grummon2025/kl_loss"]), color="gray", linestyle="--", linewidth=1.0, label="Step 0 base")

    titles = [
        (axes[0, 0], "Wu 2025 Split Cross-Loss (30-step mean)"),
        (axes[0, 1], "Wu 2025 Forward KL(p0 || p) Drift (30-step mean)"),
        (axes[0, 2], "Wu 2025 Sample Beta MSE Variance (30-step mean)"),
        (axes[1, 0], "Kriner 2026 Beta MSE (Validation Selection)"),
        (axes[1, 1], "Grummon 2025 Beta MSE (Held-Out Test)"),
        (axes[1, 2], "Grummon 2025 Human KL (Held-Out Calibration)"),
    ]
    for ax, title in titles:
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("Optimization Step")
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)

    fig.tight_layout()
    plt.show()